# GOV-01 V2 Multi-label E3: focus-weighted MobileNetV2

Purpose: train a new candidate model that gives extra importance to confirmed positive examples of crack, pothole, repaired road, and manhole cover. This notebook does not load the protected test set.

In [ ]:
# Dataset and output locations. Keep the ZIP and all outputs outside Git.
from pathlib import Path
import csv
import json
import random
import shutil
import time
import zipfile

from google.colab import drive
drive.mount('/content/drive')

DRIVE_ARCHIVE = Path('/content/drive/MyDrive/GOV-01/v2_multilabel.zip')
LOCAL_DATA_ROOT = Path('/content/v2_multilabel')
OUTPUT_DIRECTORY = Path('/content/v2_multilabel_e3_output')
CONDITIONS = ('crack', 'pothole', 'repaired_road', 'manhole_cover', 'unpaved_road', 'road_marking', 'speed_bump')
FOCUS_CONDITIONS = ('crack', 'pothole', 'repaired_road', 'manhole_cover')
FOCUS_POSITIVE_WEIGHT = 1.75
SEED = 42
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32

if not DRIVE_ARCHIVE.is_file():
    raise FileNotFoundError(f'Dataset ZIP not found: {DRIVE_ARCHIVE}')
print('Focus conditions:', FOCUS_CONDITIONS)
print('Extra positive-example weight:', FOCUS_POSITIVE_WEIGHT)
print('Protected test is deliberately not loaded by this notebook.')


In [ ]:
# Extract and verify only the train and validation data needed for this experiment.
def split_is_complete(dataset_root, split_name):
    split_root = dataset_root / split_name
    labels_path = split_root / 'labels.csv'
    if not labels_path.is_file() or not (split_root / 'images').is_dir():
        return False
    with labels_path.open(newline='', encoding='utf-8') as source:
        for row in csv.DictReader(source):
            relative_image = Path(row['materialized_image'].replace(chr(92), '/'))
            if not (split_root / relative_image).is_file():
                return False
    return True

def training_data_is_complete(dataset_root):
    return all(split_is_complete(dataset_root, name) for name in ('train', 'validation'))

if training_data_is_complete(LOCAL_DATA_ROOT):
    print('Using verified temporary Colab extraction:', LOCAL_DATA_ROOT)
else:
    if LOCAL_DATA_ROOT.exists():
        print('Removing incomplete temporary Colab extraction:', LOCAL_DATA_ROOT)
        shutil.rmtree(LOCAL_DATA_ROOT)
    print('Extracting the dataset ZIP to temporary Colab storage. This can take several minutes.')
    with zipfile.ZipFile(DRIVE_ARCHIVE) as archive:
        archive.extractall(LOCAL_DATA_ROOT.parent)
    if not training_data_is_complete(LOCAL_DATA_ROOT):
        raise RuntimeError('Extraction is incomplete. Run this cell once more; it will remove the incomplete temporary copy first.')
    print('Verified complete train and validation extraction:', LOCAL_DATA_ROOT)

OUTPUT_DIRECTORY.mkdir(parents=True, exist_ok=True)


In [ ]:
# Imports and GPU check. A GPU is required because this notebook trains a new model.
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
from sklearn.metrics import f1_score

random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)
print('TensorFlow:', tf.__version__)
print('GPU devices:', tf.config.list_physical_devices('GPU'))
if not tf.config.list_physical_devices('GPU'):
    raise RuntimeError('No GPU is connected. In Colab select Runtime, Change runtime type, T4 GPU; reconnect; then run this notebook from the first cell.')


In [ ]:
# Read labels. Unknown labels receive mask 0 and are ignored during training and scoring.
def read_split(split_name):
    split_root = LOCAL_DATA_ROOT / split_name
    paths, labels, masks = [], [], []
    with (split_root / 'labels.csv').open(newline='', encoding='utf-8') as source:
        for row in csv.DictReader(source):
            relative_image = Path(row['materialized_image'].replace(chr(92), '/'))
            image_path = split_root / relative_image
            if not image_path.is_file():
                raise FileNotFoundError(f'Missing materialized image: {image_path}')
            label_row, mask_row = [], []
            for condition in CONDITIONS:
                known = int(row[f'{condition}_known'])
                value = row[f'{condition}_present']
                if known == 0 and value != '':
                    raise ValueError(f'Unknown label has a value for {condition}: {row["record_id"]}')
                label_row.append(float(value) if known else 0.0)
                mask_row.append(float(known))
            paths.append(str(image_path))
            labels.append(label_row)
            masks.append(mask_row)
    return np.asarray(paths), np.asarray(labels, dtype='float32'), np.asarray(masks, dtype='float32')

train_paths, train_labels, train_masks = read_split('train')
validation_paths, validation_labels, validation_masks = read_split('validation')

def show_coverage(name, labels, masks):
    print(f'\n{name}: {len(labels)} images')
    for index, condition in enumerate(CONDITIONS):
        print(f'{condition:16} known={int(masks[:, index].sum()):5}  positive={int((labels[:, index] * masks[:, index]).sum()):5}')

show_coverage('Train', train_labels, train_masks)
show_coverage('Validation', validation_labels, validation_masks)


In [ ]:
# Build efficient image datasets. The model itself handles ImageNet preprocessing and training-only augmentation.
def load_image(path):
    image = tf.io.read_file(path)
    image = tf.image.decode_jpeg(image, channels=3)
    image = tf.image.resize(image, IMAGE_SIZE)
    return tf.cast(image, tf.float32)

def make_dataset(paths, labels, masks, training):
    packed_targets = np.concatenate([labels, masks], axis=1)
    dataset = tf.data.Dataset.from_tensor_slices((paths, packed_targets))
    if training:
        dataset = dataset.shuffle(min(len(paths), 4096), seed=SEED, reshuffle_each_iteration=True)
    dataset = dataset.map(lambda path, target: (load_image(path), target), num_parallel_calls=tf.data.AUTOTUNE)
    return dataset.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

train_dataset = make_dataset(train_paths, train_labels, train_masks, training=True)
validation_dataset = make_dataset(validation_paths, validation_labels, validation_masks, training=False)
validation_images = validation_dataset.map(lambda image, target: image)


In [ ]:
# This loss gives 1.75x importance only to confirmed positive labels of the four focus conditions.
FOCUS_VECTOR = tf.constant([1.0 if condition in FOCUS_CONDITIONS else 0.0 for condition in CONDITIONS], dtype=tf.float32)

@tf.keras.utils.register_keras_serializable(package='gov01')
def focus_weighted_masked_binary_crossentropy(packed_targets, probabilities):
    label_count = len(CONDITIONS)
    labels = packed_targets[:, :label_count]
    masks = packed_targets[:, label_count:]
    per_label_loss = tf.keras.backend.binary_crossentropy(labels, probabilities)
    positive_focus_bonus = labels * FOCUS_VECTOR * (FOCUS_POSITIVE_WEIGHT - 1.0)
    effective_weights = masks * (1.0 + positive_focus_bonus)
    return tf.math.divide_no_nan(tf.reduce_sum(per_label_loss * effective_weights, axis=1), tf.reduce_sum(effective_weights, axis=1))

class ValidationMaskedMacroF1(tf.keras.callbacks.Callback):
    def __init__(self, images, labels, masks):
        super().__init__()
        self.images, self.labels, self.masks = images, labels, masks
        self.history = []
    def on_epoch_end(self, epoch, logs=None):
        probabilities = self.model.predict(self.images, verbose=0)
        scores = []
        for index in range(len(CONDITIONS)):
            known = self.masks[:, index].astype(bool)
            scores.append(f1_score(self.labels[known, index], probabilities[known, index] >= 0.5, zero_division=0))
        macro_f1 = float(np.mean(scores))
        self.history.append(macro_f1)
        print(f' validation masked macro F1: {macro_f1:.4f}')


In [ ]:
# Stage 1: train a new classifier head while the ImageNet MobileNetV2 feature extractor stays frozen.
augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip('horizontal', seed=SEED),
    tf.keras.layers.RandomRotation(0.05, seed=SEED),
    tf.keras.layers.RandomContrast(0.10, seed=SEED),
], name='training_only_augmentation')
base = tf.keras.applications.MobileNetV2(include_top=False, weights='imagenet', input_shape=(*IMAGE_SIZE, 3))
base.trainable = False
inputs = tf.keras.Input(shape=(*IMAGE_SIZE, 3), name='road_image')
x = augmentation(inputs)
x = tf.keras.applications.mobilenet_v2.preprocess_input(x)
x = base(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.30, seed=SEED)(x)
outputs = tf.keras.layers.Dense(len(CONDITIONS), activation='sigmoid', name='condition_probabilities')(x)
model = tf.keras.Model(inputs, outputs, name='v2_multilabel_e3_focus_weighted_mobilenetv2')
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3), loss=focus_weighted_masked_binary_crossentropy)

head_checkpoint = OUTPUT_DIRECTORY / 'v2_multilabel_e3_head_best.keras'
head_f1 = ValidationMaskedMacroF1(validation_images, validation_labels, validation_masks)
head_history = model.fit(train_dataset, validation_data=validation_dataset, epochs=5, callbacks=[head_f1, tf.keras.callbacks.ModelCheckpoint(head_checkpoint, monitor='val_loss', save_best_only=True)], verbose=1)


In [ ]:
# Stage 2: gently fine-tune the final 30 MobileNetV2 layers, keeping Batch Normalization layers frozen.
base.trainable = True
for layer in base.layers[:-30]:
    layer.trainable = False
for layer in base.layers[-30:]:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False

model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5), loss=focus_weighted_masked_binary_crossentropy)
fine_checkpoint = OUTPUT_DIRECTORY / 'v2_multilabel_e3_focus_weighted_best.keras'
fine_f1 = ValidationMaskedMacroF1(validation_images, validation_labels, validation_masks)
start_time = time.time()
fine_history = model.fit(train_dataset, validation_data=validation_dataset, epochs=15, callbacks=[fine_f1, tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=4, restore_best_weights=True), tf.keras.callbacks.ModelCheckpoint(fine_checkpoint, monitor='val_loss', save_best_only=True)], verbose=1)
print(f'Fine-tuning time: {time.time() - start_time:.1f} seconds')


In [ ]:
# Save validation evidence and learning curves. Compare these results with E2 before any protected-test run.
best_model = tf.keras.models.load_model(fine_checkpoint, compile=False)
validation_probabilities = best_model.predict(validation_images, verbose=1)
per_condition_f1 = {}
for index, condition in enumerate(CONDITIONS):
    known = validation_masks[:, index].astype(bool)
    per_condition_f1[condition] = float(f1_score(validation_labels[known, index], validation_probabilities[known, index] >= 0.5, zero_division=0))

metrics = {
    'experiment': 'v2_multilabel_e3_focus_weighted_mobilenetv2',
    'selection_data': 'validation only',
    'protected_test_loaded': False,
    'focus_conditions': list(FOCUS_CONDITIONS),
    'focus_positive_weight': FOCUS_POSITIVE_WEIGHT,
    'threshold': 0.5,
    'masked_validation_macro_f1': float(np.mean(list(per_condition_f1.values()))),
    'per_condition_f1': per_condition_f1,
    'head_epochs_completed': len(head_history.history['loss']),
    'fine_tune_epochs_completed': len(fine_history.history['loss']),
}
(OUTPUT_DIRECTORY / 'v2_multilabel_e3_validation_metrics.json').write_text(json.dumps(metrics, indent=2) + '\n')
best_model.save(OUTPUT_DIRECTORY / 'v2_multilabel_e3_focus_weighted_final.keras')
print(json.dumps(metrics, indent=2))

plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(head_history.history['loss'] + fine_history.history['loss'], label='train loss')
plt.plot(head_history.history['val_loss'] + fine_history.history['val_loss'], label='validation loss')
plt.axvline(len(head_history.history['loss']) - 0.5, color='gray', linestyle='--', label='fine-tuning starts')
plt.xlabel('epoch')
plt.ylabel('focus-weighted masked loss')
plt.legend()
plt.subplot(1, 2, 2)
plt.plot(head_f1.history + fine_f1.history, label='validation masked macro F1')
plt.xlabel('epoch')
plt.ylabel('F1')
plt.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIRECTORY / 'v2_multilabel_e3_learning_curve.png', dpi=150)
plt.show()
print('Validation-only evidence saved. The protected test remains untouched.')


In [ ]:
# Download the model and validation evidence after you have read the results. Do not add this ZIP to Git.
from google.colab import files
result_zip = Path('/content/v2_multilabel_e3_output.zip')
if result_zip.exists():
    result_zip.unlink()
shutil.make_archive(str(result_zip.with_suffix('')), 'zip', OUTPUT_DIRECTORY)
print('Downloading:', result_zip.name)
files.download(str(result_zip))
